## TODO
- [X] GridSearchCV
- [X] Add PCA
- [X] PCA Feature Loadings
- [ ] Try SMOTE and SMOTE Tomek
- [ ] Write PCA Result Tables

Current layout is a bit messy. Maybe test multiple models in different stages:
1. 4 train/test split types + gridsearchCV
2. Best* train/test model + PCA tuning across 5 component counts
3. Best PCA model vs. Model with no PCA
4. For the end model, show:
- What train/test split
- What hyperparameters
- PCA Components (if used)
- Classification Report
- ROC AUC
- SHAP Values OR Loadings (if PCA was used)

\* - At each stage, judge the 'best' on the classification report (specifically recall)

# Thyroid Cancer Recurrence: Random Forest Classification
**Author:** Justin Williams  
**Date:** 11/3/2026

A demonstration of the Random Forest classification model. The notebook compares two evaluation methods: train/test splits and 5-fold cross-validation.

## Contents

1. Background
2. Setup and Constants
3. Load Data
4. Preprocessing
5. Model Pipeline
6. Evaluation
7. Results


## 1. Background

A `RandomForestClassifier` is an ensemble of decision trees. Each tree is trained on a random sample of rows and considers a random selection of features at each split. Each tree votes, and the majority vote is chosen for the final prediction.



## 2. Setup and Constants

In [59]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, 
    confusion_matrix,
    roc_curve,
    auc
)
from sklearn import preprocessing
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold, 
    cross_val_predict, 
    train_test_split
)

In [60]:
TARGET_COL = "Recurred"
CATEGORICAL_COLS = [
    "Gender",
    "Smoking",
    "Hx Smoking",
    "Hx Radiotherapy",
    "Thyroid Function",
    "Physical Examination",
    "Adenopathy",
    "Pathology",
    "Focality",
    "Risk",
    "T",
    "N",
    "M",
    "Stage",
    "Response"
]

RAND_STATE = 42

TEST_SIZES = [0.25, 0.3, 0.35]

PARAM_DICT = {
    "clf__n_estimators": [80, 85, 90],
    "clf__max_features": ["sqrt", "log2"],
    "clf__max_depth": [10, 15, 20]
}

N_COMPONENTS = 21

pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", '{:.3f}'.format)

In [61]:
# Format a confusion matrix (cm) into a Pandas dataframe
def make_cm_frame(cm):
    return pd.DataFrame(
        cm,
        index=["Actual No", "Actual Yes"],
        columns=["Pred No", "Pred Yes"]
    )

def make_load_frame(col_names, loadings):
    return pd.DataFrame(loadings)


### Cross-Validation Setup

`StratifiedKFold` keeps the ratio of target values consistent across each fold.

In [62]:
# Uses StratifiedKFold to keep the ratio of target values consistent across folds
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RAND_STATE)

## 3. Load Data

In [63]:
df = pd.read_csv("../data/processed/Thyroid_Diff.csv")

print(df.shape)

df.head()

(383, 17)


,Age,Gender,Smoking,Hx Smoking,Hx Radiotherapy,Thyroid Function,Physical Examination,Adenopathy,Pathology,Focality,Risk,T,N,M,Stage,Response,Recurred
0,27,F,No,No,No,Euthyroid,Single nodular goiter-left,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Indeterminate,No
1,34,F,No,Yes,No,Euthyroid,Multinodular goiter,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
2,30,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
3,62,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
4,62,F,No,No,No,Euthyroid,Multinodular goiter,No,Micropapillary,Multi-Focal,Low,T1a,N0,M0,I,Excellent,No


In [64]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 383 entries, 0 to 382
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Age                   383 non-null    int64
 1   Gender                383 non-null    str  
 2   Smoking               383 non-null    str  
 3   Hx Smoking            383 non-null    str  
 4   Hx Radiotherapy       383 non-null    str  
 5   Thyroid Function      383 non-null    str  
 6   Physical Examination  383 non-null    str  
 7   Adenopathy            383 non-null    str  
 8   Pathology             383 non-null    str  
 9   Focality              383 non-null    str  
 10  Risk                  383 non-null    str  
 11  T                     383 non-null    str  
 12  N                     383 non-null    str  
 13  M                     383 non-null    str  
 14  Stage                 383 non-null    str  
 15  Response              383 non-null    str  
 16  Recurred           

## 4. Preprocessing

`X` holds the features to train on. `y` holds the target to predict. The target column maps Yes/No to 1/0.

In [65]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].map({"No": 0, "Yes": 1})

print(y.value_counts())

Recurred
0    275
1    108
Name: count, dtype: int64


### Encoding and Scaling

Columns listed in `CATEGORICAL_COLS` are transformed using One-Hot Encoding. The `Age` column is scaled to have a mean of 0 and a standard deviation of 1. The `ColumnTransformer` is used to avoid data leakage from encoding training and testing data together. It produces the same columns, even if future data is missing any categories.


In [66]:
colTransform = ColumnTransformer(
    transformers=[
        ("cat", preprocessing.OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
        ("num", preprocessing.StandardScaler(), ["Age"])
    ],
    verbose_feature_names_out=False
)

## 5. Pipeline Setup

In [67]:
model = Pipeline([
    ("trans", colTransform),
    ("pca", PCA(n_components=PCA_COMPONENTS)),
    ("clf", RandomForestClassifier(random_state=RAND_STATE))
])

model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('trans', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{

### Hyperparameter Tuning - GridSearchCV

Adjust the parameters of the Random Forest classifier in the pipeline to find the combination of parameters with the best results. Fit to each train/test split.

In [68]:
gs = GridSearchCV(model, PARAM_DICT)

## 6. Evaluation

### Train-Test Splits

In [69]:
results = []
split_cms = {}

for size in TEST_SIZES:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=size, stratify=y, random_state=RAND_STATE
    )

    gs = GridSearchCV(model, PARAM_DICT)
    gs.fit(X_train, y_train)
    y_pred = gs.predict(X_test)

    fpr, tpr, thresholds = roc_curve(y_test, y_pred)
    roc_auc = auc(fpr, tpr)

    enc = gs.best_estimator_.named_steps["trans"]
    pca = gs.best_estimator_.named_steps["pca"]
    loadings = pd.DataFrame(
        (pca.components_.T * np.sqrt(pca.explained_variance_)),
        index=enc.get_feature_names_out(),
        columns=(f"PC{i+1}" for i in range(pca.n_components_))
    )

    results.append({
        "Method": f"Train/test split ({size})",
        "Report": classification_report(y_test, y_pred, target_names=["No", "Yes"], output_dict=True),
        "ROCAUC": roc_auc,
        "Best": gs.best_params_,
        "FeatLoad": loadings
    })
    split_cms[size] = confusion_matrix(y_test, y_pred)

### 5-Fold Cross-Validation

In [ ]:
folds = []

for i_train, i_test in kfold.split(X, y):
    X_train, X_test, y_train, y_test = X.iloc[i_train], X.iloc[i_test], y.iloc[i_train], y.iloc[i_test]
    gs.fit(X_train, y_train)

    enc = gs.best_estimator_.named_steps["trans"]
    pca = gs.best_estimator_.named_steps["pca"]
    loadings = pd.DataFrame(
        (pca.components_.T * np.sqrt(pca.explained_variance_)),
        index=enc.get_feature_names_out(),
        columns=(f"PC{i+1}" for i in range(pca.n_components_))
    )

    folds.append({
        "i_train": i_train,
        "i_test": i_test,
        "acc": gs.score(X_test, y_test),
        "pred": gs.predict(X_test),
        "test": y_test,
        "params": gs.best_params_,
        "loadings": loadings
    })

fold_scores = [f.get("acc") for f in folds]

high = max(fold_scores)
high_fold = folds[fold_scores.index(high)]
low = min(fold_scores)
avg = np.mean(fold_scores)

cv_cm = confusion_matrix(high_fold["test"], high_fold["pred"])

fpr, tpr, thresholds = roc_curve(high_fold["test"], high_fold["pred"])
roc_auc = auc(fpr, tpr)
    
results.append({
    "Method": "5-fold cross-validation (max)",
    "Report": classification_report(high_fold["test"], high_fold["pred"], target_names=["No", "Yes"], output_dict=True),
    "ROCAUC": roc_auc,
    "Best": high_fold["params"],
    "FeatLoad": high_fold["loadings"]
})



KFold Scores: [<generator object <genexpr> at 0x0000025E1186A260>0.88]
KFold Average: 0.8799043062200956


In [82]:
print(f"KFold Scores: {fold_scores}")
print(f"KFold Average: {avg}")

KFold Scores: [0.8311688311688312, 0.9090909090909091, 0.8961038961038961, 0.881578947368421, 0.881578947368421]
KFold Average: 0.8799043062200956


## 7. Results

### Classification Reports

In [75]:
for result in results:
    print(result["Method"])
    print(f"ROC AUC: {result["ROCAUC"]}")
    display(pd.DataFrame(result["Report"]))


Train/test split (0.25)
ROC AUC: 0.8115942028985507


,No,Yes,accuracy,macro avg,weighted avg
precision,0.880,0.857,0.875,0.869,0.874
recall,0.957,0.667,0.875,0.812,0.875
f1-score,0.917,0.750,0.875,0.833,0.870
support,69.000,27.000,0.875,96.000,96.000


Train/test split (0.3)
ROC AUC: 0.8676581325301205


,No,Yes,accuracy,macro avg,weighted avg
precision,0.937,0.750,0.878,0.843,0.885
recall,0.892,0.844,0.878,0.868,0.878
f1-score,0.914,0.794,0.878,0.854,0.880
support,83.000,32.000,0.878,115.000,115.000


Train/test split (0.35)
ROC AUC: 0.8323385784047749


,No,Yes,accuracy,macro avg,weighted avg
precision,0.900,0.800,0.874,0.850,0.872
recall,0.928,0.737,0.874,0.832,0.874
f1-score,0.914,0.767,0.874,0.840,0.872
support,97.000,38.000,0.874,135.000,135.000


5-fold cross-validation (max)
ROC AUC: 0.8954545454545455


,No,Yes,accuracy,macro avg,weighted avg
precision,0.944,0.826,0.909,0.885,0.911
recall,0.927,0.864,0.909,0.895,0.909
f1-score,0.936,0.844,0.909,0.890,0.910
support,55.000,22.000,0.909,77.000,77.000


### Confusion Matrices

The rows represent the actual classes, and the columns represent predicted classes. All correct predictions lie on the diagonal:

| | Pred No | Pred Yes |
|---|---|---|
| **Actual No** | True negatives (TN) | False positives (FP) |
| **Actual Yes** | False negatives (FN) | True positives (TP) |

Train/Test Split matrices count the test set while the Cross-Validation matrix counts the entire dataset.

In [83]:
for size, cm in split_cms.items():
    print(f"Confusion matrix (test_size={size}):")
    display(make_cm_frame(cm))

print("Confusion matrix (5-fold CV):")
display(make_cm_frame(cv_cm))

Confusion matrix (test_size=0.25):


,Pred No,Pred Yes
Actual No,66,3
Actual Yes,9,18


Confusion matrix (test_size=0.3):


,Pred No,Pred Yes
Actual No,74,9
Actual Yes,5,27


Confusion matrix (test_size=0.35):


,Pred No,Pred Yes
Actual No,90,7
Actual Yes,10,28


Confusion matrix (5-fold CV):


,Pred No,Pred Yes
Actual No,51,4
Actual Yes,3,19


### Hyperparameter Tuning - GridSearchCV

The following hyper parameters for the Random Forest model were tuned:
1. `max_depth` - How deep each tree can go
2. `max_features` - Total number of features each tree can train on
3. `n_estimators` - Number of trees voting


In [84]:
for result in results:
    print(result["Method"])
    print(result["Best"] if result["Best"] else "N/A")
    print()

Train/test split (0.25)
{'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__n_estimators': 80}

Train/test split (0.3)
{'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__n_estimators': 85}

Train/test split (0.35)
{'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__n_estimators': 80}

5-fold cross-validation (max)
{'clf__max_depth': 15, 'clf__max_features': 'sqrt', 'clf__n_estimators': 80}



### Feature Loadings

Matrix of feature loadings, explaining the impact of each feature across each component.


In [85]:
for result in results:
    print(result["Method"])
    display(result["FeatLoad"])

Train/test split (0.25)


,PC1,PC2
Gender_F,-0.185,0.005
Gender_M,0.185,-0.005
Smoking_No,-0.175,-0.038
Smoking_Yes,0.175,0.038
Hx Smoking_No,-0.042,-0.037
Hx Smoking_Yes,0.042,0.037
Hx Radiotherapy_No,-0.029,-0.020
Hx Radiotherapy_Yes,0.029,0.020
Thyroid Function_Clinical Hyperthyroidism,-0.001,0.024
Thyroid Function_Clinical Hypothyroidism,-0.014,-0.006


Train/test split (0.3)


,PC1,PC2
Gender_F,-0.171,0.011
Gender_M,0.171,-0.011
Smoking_No,-0.167,-0.034
Smoking_Yes,0.167,0.034
Hx Smoking_No,-0.035,-0.038
Hx Smoking_Yes,0.035,0.038
Hx Radiotherapy_No,-0.031,-0.021
Hx Radiotherapy_Yes,0.031,0.021
Thyroid Function_Clinical Hyperthyroidism,-0.000,0.027
Thyroid Function_Clinical Hypothyroidism,-0.015,-0.006


Train/test split (0.35)


,PC1,PC2
Gender_F,-0.147,0.006
Gender_M,0.147,-0.006
Smoking_No,-0.142,-0.032
Smoking_Yes,0.142,0.032
Hx Smoking_No,-0.017,-0.031
Hx Smoking_Yes,0.017,0.031
Hx Radiotherapy_No,-0.003,-0.007
Hx Radiotherapy_Yes,0.003,0.007
Thyroid Function_Clinical Hyperthyroidism,-0.016,0.007
Thyroid Function_Clinical Hypothyroidism,-0.017,-0.006


5-fold cross-validation (max)


,PC1,PC2
Gender_F,-0.172,0.013
Gender_M,0.172,-0.013
Smoking_No,-0.166,-0.029
Smoking_Yes,0.166,0.029
Hx Smoking_No,-0.031,-0.017
Hx Smoking_Yes,0.031,0.017
Hx Radiotherapy_No,-0.027,-0.009
Hx Radiotherapy_Yes,0.027,0.009
Thyroid Function_Clinical Hyperthyroidism,-0.008,0.026
Thyroid Function_Clinical Hypothyroidism,-0.011,0.006
